In [ ]:
import pickle

# Load embedding
# review collaborative embeddings: "../../data/games_with_embeddings.pkl"
# game description embeddings: "../../data/game_descriptions_embeddings.pkl"
with open("../../data/games_with_embeddings.pkl", "rb") as f:
    game_embedding_dict = pickle.load(f)

In [2]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def find_closest_games(target_game, embeddings_dict, top_n=5):
    """
    Finds the most similar games based on their embedding vectors.
    
    :param target_game: Name of the game to search for (str)
    :param embeddings_dict: Dictionary mapping Game Name -> Numpy Array (size 32)
    :param top_n: Number of recommendations to return
    """
    
    # Check if the game exists in our database
    if target_game not in embeddings_dict:
        return f"Error: '{target_game}' not found in the embeddings database."

    # Isolate the target vector and reshape it for sklearn
    # reshape(1, -1) turns it from shape (32,) to (1, 32)
    target_vector = embeddings_dict[target_game].reshape(1, -1)
    
    # Prepare the rest of the data
    # We separate names and vectors so their indexes match up
    game_names = list(embeddings_dict.keys())
    all_vectors = np.array(list(embeddings_dict.values()))
    
    # Calculate Cosine Similarity
    # This compares the target (1, 32) against all games (N, 32) simultaneously
    # It returns an array of scores from -1.0 (opposites) to 1.0 (identical)
    similarity_scores = cosine_similarity(target_vector, all_vectors)[0]
    
    # Sort the results
    # argsort() gives us the indexes from lowest to highest, so we reverse it [::-1]
    ranked_indexes = np.argsort(similarity_scores)[::-1]
    
    # Format the output
    print(f"Games most similar to '{target_game}':\n")
    results = []
    
    for idx in ranked_indexes:
        match_name = game_names[idx]
        score = similarity_scores[idx]
        
        # Skip the target game itself (it will always have a 1.0 score)
        if match_name == target_game:
            continue
            
        results.append((match_name, score))
        print(f"{len(results)}. {match_name} (Similarity Score: {score:.4f})")
        
        # Stop once we hit our desired number of recommendations
        if len(results) == top_n:
            break
            
    return results

In [22]:
# For reproducible random numbers
np.random.seed(42) 

# Run the function
find_closest_games("Marvel's Spider-Man 2", game_embedding_dict, top_n=10)

Games most similar to 'Marvel's Spider-Man 2':

1. Ghost of Tsushima: Director's Cut (Similarity Score: 0.9659)
2. Metal Eden (Similarity Score: 0.9631)
3. God of War: Ragnarok - Valhalla (Similarity Score: 0.9610)
4. Destroy All Humans! 2 - Reprobed (Similarity Score: 0.9602)
5. F1 22 (Similarity Score: 0.9593)
6. Death Stranding 2: On The Beach (Similarity Score: 0.9593)
7. The Last of Us Part II Remastered (Similarity Score: 0.9589)
8. Eternights (Similarity Score: 0.9587)
9. The Last of Us Part I (Similarity Score: 0.9572)
10. God of War: Ragnarok (Similarity Score: 0.9562)


[("Ghost of Tsushima: Director's Cut", np.float64(0.9658834795152069)),
 ('Metal Eden', np.float64(0.963113523551566)),
 ('God of War: Ragnarok - Valhalla', np.float64(0.9609940370004291)),
 ('Destroy All Humans! 2 - Reprobed', np.float64(0.9602159965955992)),
 ('F1 22', np.float64(0.9593333262850459)),
 ('Death Stranding 2: On The Beach', np.float64(0.9592892944986081)),
 ('The Last of Us Part II Remastered', np.float64(0.9588924359918091)),
 ('Eternights', np.float64(0.9586983894881638)),
 ('The Last of Us Part I', np.float64(0.9572074655935046)),
 ('God of War: Ragnarok', np.float64(0.9561772152038336))]